# **Baseball Salary Analysis (Assignment 1)**

**Purpose:** Identify and explain 10 interesting findings about MLB player salaries using at least five analytic methods (descriptive statistics, charts, crosstabs, correlation, hypothesis tests, ANOVA, and regression).

## 1. Setup

In [ ]:
from pathlib import Path
import re
import shutil
import warnings
import logging
import textwrap
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# stats / models
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Docx helper (optional)
try:
    from docx import Document
except Exception:
    Document = None

# Colab file utilities (Colab only)
try:
    from google.colab import files
except Exception:
    files = None

# Silence noisy logs (matplotlib + other libs)
logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# -----------------------------
# Output folders
# -----------------------------
BASE_OUT = Path("outputs")
DIRS = {
    "tables": BASE_OUT / "tables",
    "figures": BASE_OUT / "figures",
    "models": BASE_OUT / "models",
    "appendix": BASE_OUT / "appendix",
}

RESET_OUTPUTS_EACH_RUN = True
if RESET_OUTPUTS_EACH_RUN and BASE_OUT.exists():
    shutil.rmtree(BASE_OUT)

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.grid": False,
})

# -----------------------------
# Counters (file naming only)
# -----------------------------
TABLE_NO = 0
APP_TABLE_NO = 0
FIG_NO = 0
MODEL_NO = 0

# -----------------------------
# Utility helpers
# -----------------------------
def _slugify(text: str, max_len: int = 90) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text)
    return text[:max_len].strip("_") or "output"

def _ensure_suffix(name: str, suffix: str) -> str:
    return name if name.lower().endswith(suffix.lower()) else f"{name}{suffix}"

# -----------------------------
# Presentation labels (NO underscores)
# -----------------------------
def pretty_label(x: str) -> str:
    """
    Presentation-friendly labels:
    - Replace underscores with spaces
    - Keep all-caps fields as all-caps
    - Light cleanup for common baseball/stat tokens
    """
    s = "" if x is None else str(x)
    s = s.replace("_", " ").strip()
    s = re.sub(r"\s+", " ", s)

    replacements = {
        "NO ATBAT": "NO AT-BAT",
        "NO HOME": "NO HOME RUNS",
        "CR HOME": "CR HOME RUNS",
        "LOG SALARY": "LOG(SALARY)",
        "PVAL": "P-VALUE",
        "P VALUE": "P-VALUE",
        "TEAM SUGGESTION": "TEAM SUGGESTION",
        "ISSUE NOTE": "ISSUE NOTE",
    }
    s_up = s.upper()
    if s_up in replacements:
        return replacements[s_up]

    compact = re.sub(r"[ \d\-\(\)]", "", s)
    if compact.isupper():
        return s

    return s.title()

# -----------------------------
# Formatting helpers
# -----------------------------
def _is_whole_number(x, tol=1e-9):
    try:
        xf = float(x)
        return abs(xf - round(xf)) < tol
    except Exception:
        return False

def format_currency(x):
    """Currency: $875 if whole, else $875.25"""
    if pd.isna(x):
        return ""
    try:
        xf = float(x)
    except Exception:
        return str(x)
    if _is_whole_number(xf):
        return f"${int(round(xf)):,}"
    return f"${xf:,.2f}".rstrip("0").rstrip(".")

def format_integer(x):
    if pd.isna(x):
        return ""
    try:
        return f"{int(round(float(x))):,}"
    except Exception:
        return str(x)

def format_fixed(x, decimals=2):
    """Numeric: up to decimals, drop trailing .00, and show whole numbers without decimals."""
    if pd.isna(x):
        return ""
    try:
        xf = float(x)
    except Exception:
        return str(x)
    if _is_whole_number(xf):
        return f"{int(round(xf)):,}"
    s = f"{round(xf, decimals):,.{decimals}f}"
    return s.rstrip("0").rstrip(".")

def format_corr(x):
    return format_fixed(x, decimals=3)

def format_pval(x):
    if pd.isna(x):
        return ""
    try:
        xf = float(x)
    except Exception:
        return str(x)
    if xf < 0.0001:
        return "0.0000"
    return format_fixed(xf, decimals=4)

# -----------------------------
# DataFrame -> string table
# -----------------------------
def _df_to_string_table(
    df: pd.DataFrame,
    *,
    decimals: int = 2,
    currency_cols=None,
    int_cols=None,
    corr_cols=None,
    pval_cols=None,
    extra_decimal_map=None,
):
    currency_cols = set(currency_cols or [])
    int_cols = set(int_cols or [])
    corr_cols = set(corr_cols or [])
    pval_cols = set(pval_cols or [])
    extra_decimal_map = extra_decimal_map or {}

    out = df.copy()
    for col in out.columns:
        if col in currency_cols:
            out[col] = out[col].apply(format_currency)
        elif col in int_cols:
            out[col] = out[col].apply(format_integer)
        elif col in corr_cols:
            out[col] = out[col].apply(format_corr)
        elif col in pval_cols:
            out[col] = out[col].apply(format_pval)
        elif col in extra_decimal_map:
            out[col] = out[col].apply(lambda v: format_fixed(v, decimals=extra_decimal_map[col]))
        elif pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].apply(lambda v: format_fixed(v, decimals))
        else:
            out[col] = out[col].astype(object).where(out[col].notna(), "")
            out[col] = out[col].astype(str).replace({"nan": "", "NaN": "", "None": ""})

    out.index = out.index.astype(str)
    return out

# -----------------------------
# Column widths + style parsing
# -----------------------------
def _header_visual_len(s: str) -> int:
    lines = str(s).split("\n")
    return max((len(x) for x in lines), default=len(str(s)))

def _cell_visual_len(s: str) -> int:
    lines = str(s).split("\n")
    return max((len(x) for x in lines), default=len(str(s)))

def _compute_colwidths(
    cell_text: pd.DataFrame,
    col_labels,
    include_index: bool,
    *,
    min_chars_override: dict | None = None,
) -> list:
    """
    Compute relative col widths with minimum width protection for key columns.
    Keys are DISPLAY labels (spaces, no underscores).
    """
    MIN_CHARS = {
        "NAME": 14,
        "TEAM": 10,
        "LEAGUE": 10,
        "DIVISION": 10,
        "POSITION": 10,
        "SALARY": 11,
        "TEAM SUGGESTION": 20,
        "ISSUE NOTE": 44,
        "UNKNOWN POSITIONS": 22,
    }
    min_chars_override = {str(k).upper(): int(v) for k, v in (min_chars_override or {}).items()}

    max_lens = []

    if include_index:
        idx_max = max([len(str(x)) for x in cell_text.index] + [4])
        max_lens.append(max(4, idx_max))

    for c in col_labels:
        c_str = str(c)
        c_key = c_str.upper()

        header_len = _header_visual_len(c_str)
        body_len = 0
        for v in cell_text[c].astype(str).tolist():
            body_len = max(body_len, _cell_visual_len(v))

        m = max(header_len, body_len)
        m = max(m, min_chars_override.get(c_key, MIN_CHARS.get(c_key, 6)))

        if c_key == "ISSUE NOTE":
            m = int(m * 1.15)
        elif c_key == "NAME":
            m = int(m * 1.10)

        max_lens.append(m)

    total = sum(max_lens) if sum(max_lens) > 0 else 1
    return [m / total for m in max_lens]

def _parse_css_style(style_str: str) -> dict:
    out = {}
    if not style_str:
        return out
    parts = [p.strip() for p in style_str.split(";") if p.strip()]
    for p in parts:
        if ":" not in p:
            continue
        k, v = p.split(":", 1)
        out[k.strip().lower()] = v.strip()
    return out

# -----------------------------
# Render table PNG
# -----------------------------
def _render_table_png(
    df_str: pd.DataFrame,
    caption: str,
    out_path: Path,
    *,
    include_index: bool = True,
    max_width_in: float = 34.0,
    base_fontsize: int = 9,
    style_df: pd.DataFrame | None = None,
    header_height_mult: float = 1.25,
    cell_pad: float = 0.08,
    row_scale: float | None = None,
    min_chars_override: dict | None = None,
):
    n_rows, n_cols = df_str.shape

    width_in = min(max_width_in, max(12.0, 0.95 * (n_cols + (1 if include_index else 0)) + 8.0))

    if n_rows <= 22:
        height_in = max(4.0, 0.38 * (n_rows + 4))
    elif n_rows <= 45:
        height_in = max(6.0, 0.30 * (n_rows + 4))
    else:
        height_in = max(7.5, 0.24 * (n_rows + 4))
    height_in = min(22.0, height_in)

    fig, ax = plt.subplots(figsize=(width_in, height_in))
    ax.axis("off")

    col_labels = list(df_str.columns)

    if include_index:
        display_cols = [""] + col_labels
        cell_rows = [[df_str.index[i]] + df_str.iloc[i].tolist() for i in range(n_rows)]
    else:
        display_cols = col_labels
        cell_rows = df_str.values.tolist()

    col_widths = _compute_colwidths(
        df_str,
        col_labels,
        include_index,
        min_chars_override=min_chars_override,
    )

    tbl = ax.table(
        cellText=cell_rows,
        colLabels=display_cols,
        cellLoc="center",
        colLoc="center",
        loc="upper left",
        colWidths=col_widths,
    )

    # padding
    for (r, c), cell in tbl.get_celld().items():
        try:
            cell.PAD = float(cell_pad)
        except Exception:
            pass

    # font + scaling
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(base_fontsize)

    if row_scale is None:
        if n_rows <= 22:
            row_scale = 1.30
        elif n_rows <= 45:
            row_scale = 1.10
        else:
            row_scale = 0.98
    tbl.scale(1.0, float(row_scale))

    # base styling
    for (r, c), cell in tbl.get_celld().items():
        cell.set_linewidth(0.35)
        cell.set_edgecolor("#B0B0B0")
        if r == 0:
            cell.set_facecolor("#E6E6E6")
            cell.set_text_props(weight="bold")
            cell.set_linewidth(0.8)
            cell.set_edgecolor("#333333")
        if include_index and c == 0 and r > 0:
            cell.set_facecolor("#F2F2F2")

    # header row height
    for c in range(len(display_cols)):
        key = (0, c)
        if key in tbl.get_celld():
            tbl[key].set_height(tbl[key].get_height() * float(header_height_mult))
            tbl[key].get_text().set_va("center")

    # per-cell styles (keeps your colors)
    if style_df is not None:
        for i in range(n_rows):
            for j, col in enumerate(df_str.columns):
                try:
                    style_str = style_df.iloc[i, style_df.columns.get_loc(col)]
                except Exception:
                    style_str = ""
                rules = _parse_css_style(style_str)

                r = i + 1
                c = j + (1 if include_index else 0)
                cell = tbl.get_celld().get((r, c))
                if cell is None:
                    continue

                bg = rules.get("background-color")
                if bg:
                    cell.set_facecolor(bg)

                fw = rules.get("font-weight")
                if fw and fw.strip() in ["700", "bold", "bolder"]:
                    cell.get_text().set_weight("bold")

                ta = rules.get("text-align")
                if ta:
                    ta = ta.strip().lower()
                    if ta in ["left", "center", "right"]:
                        cell.get_text().set_ha(ta)

                color = rules.get("color")
                if color:
                    cell.get_text().set_color(color)

                if rules.get("border-left"):
                    cell.set_edgecolor("#333333")
                    cell.set_linewidth(1.2)

    # caption centered and close
    ax.text(
        0.5, 1, caption,
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=10, fontweight="normal"
    )

    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    plt.close(fig)

# -----------------------------
# Display + save titled table (NO individual downloads)
# -----------------------------
def display_titled_table(
    df: pd.DataFrame,
    title: str,
    filename: str,
    *,
    folder: str = "tables",
    currency_cols=None,
    decimals: int = 2,
    index: bool = True,
    extra_style=None,
    int_cols=None,
    corr_cols=None,
    pval_cols=None,
    extra_decimal_map=None,
    wrap_cols=None,
    wrap_width: int = 48,
    max_width_in: float = 30.0,
    header_wrap_map: dict | None = None,
    header_height_mult: float = 1.25,
    cell_pad: float = 0.08,
    row_scale: float | None = None,
    min_chars_override: dict | None = None,
):
    """
    Displays in notebook and saves CSV/PNG/HTML to outputs/.
    DOES NOT download anything individually (zip-only export happens later).
    """
    global TABLE_NO, APP_TABLE_NO

    if folder == "appendix":
        APP_TABLE_NO += 1
        file_prefix = f"A{APP_TABLE_NO:02d}_"
    else:
        TABLE_NO += 1
        file_prefix = f"{TABLE_NO:02d}_"

    caption = f"{title}"

    out_dir = DIRS.get(folder, DIRS["tables"])
    base = file_prefix + _slugify(filename)

    csv_path = out_dir / _ensure_suffix(base, ".csv")
    png_path = out_dir / _ensure_suffix(base, ".png")
    html_path = out_dir / _ensure_suffix(base, ".html")

    # raw CSV
    df.to_csv(csv_path, index=index)

    # prepare DF
    df_out = df.copy()
    if not index:
        df_out = df_out.reset_index(drop=True)

    # wrap selected text columns
    wrap_cols = set(wrap_cols or [])
    if wrap_cols:
        def _wrap_text(s, width):
            s = "" if pd.isna(s) else str(s)
            if len(s) <= width:
                return s
            words = s.split(" ")
            lines, cur = [], ""
            for w in words:
                if len(cur) + len(w) + 1 <= width:
                    cur = (cur + " " + w).strip()
                else:
                    if cur:
                        lines.append(cur)
                    cur = w
            if cur:
                lines.append(cur)
            return "\n".join(lines)

        for c in wrap_cols:
            if c in df_out.columns:
                df_out[c] = df_out[c].apply(lambda v: _wrap_text(v, wrap_width))

    # display-friendly copy
    df_disp = df_out.copy()
    df_disp.columns = [pretty_label(c) for c in df_disp.columns]
    df_disp.index = [pretty_label(i) for i in df_disp.index]

    # optional header wrap
    wrapped_header_map = None
    if header_wrap_map:
        wrapped_header_map = {pretty_label(k): v for k, v in header_wrap_map.items()}
        df_disp.columns = [wrapped_header_map.get(c, c) for c in df_disp.columns]

    # display in notebook (centered)
    display(HTML(
        f"<div style='font-weight:700; font-size:13px; margin:6px 0 6px 0; text-align:center;'>{caption}</div>"
    ))
    display(df_disp if index else df_disp.reset_index(drop=True))

    # formatting lists aligned to df_disp
    currency_cols_disp = [pretty_label(c) for c in (currency_cols or [])]
    int_cols_disp = [pretty_label(c) for c in (int_cols or [])]
    corr_cols_disp = [pretty_label(c) for c in (corr_cols or [])]
    pval_cols_disp = [pretty_label(c) for c in (pval_cols or [])]
    extra_decimal_map_disp = {pretty_label(k): v for k, v in (extra_decimal_map or {}).items()}

    df_str = _df_to_string_table(
        df_disp,
        decimals=decimals,
        currency_cols=currency_cols_disp,
        int_cols=int_cols_disp,
        corr_cols=corr_cols_disp,
        pval_cols=pval_cols_disp,
        extra_decimal_map=extra_decimal_map_disp,
    )

    # optional style DF
    style_df = None
    if callable(extra_style):
        try:
            style_df = extra_style(df_out)
            if isinstance(style_df, pd.DataFrame):
                style_df = style_df.copy()
                style_df.columns = [pretty_label(c) for c in style_df.columns]
                style_df.index = [pretty_label(i) for i in style_df.index]
                if wrapped_header_map:
                    style_df.columns = [wrapped_header_map.get(c, c) for c in style_df.columns]
                style_df = style_df.reindex(index=df_disp.index, columns=df_disp.columns).fillna("")
            else:
                style_df = None
        except Exception:
            style_df = None

    # font sizing for very wide tables
    ncols_for_font = df_str.shape[1] + (1 if index else 0)
    fontsize = 9
    if ncols_for_font >= 12:
        fontsize = 8
    if ncols_for_font >= 18:
        fontsize = 7

    _render_table_png(
        df_str,
        caption=caption,
        out_path=png_path,
        include_index=index,
        max_width_in=max_width_in,
        base_fontsize=fontsize,
        style_df=style_df,
        header_height_mult=header_height_mult,
        cell_pad=cell_pad,
        row_scale=row_scale,
        min_chars_override=min_chars_override,
    )

    # HTML snapshot
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(f"<h3>{caption}</h3>\n")
        f.write(df_disp.to_html(index=index, escape=False))

    return {"csv": csv_path, "png": png_path, "html": html_path}

# -----------------------------
# Figure saver (NO individual downloads)
# -----------------------------
def _prettify_ticklabels(ax):
    xt = ax.get_xticklabels()
    if any(("_" in t.get_text()) for t in xt):
        ax.set_xticklabels([pretty_label(t.get_text()) for t in xt])

    yt = ax.get_yticklabels()
    if any(("_" in t.get_text()) for t in yt):
        ax.set_yticklabels([pretty_label(t.get_text()) for t in yt])

def save_figure(
    title: str,
    filename: str,
    *,
    folder: str = "figures",
    fig=None,
    ax=None,
    show: bool = True,
):
    global FIG_NO
    FIG_NO += 1

    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = plt.gca()

    ax.set_xlabel(pretty_label(ax.get_xlabel()))
    ax.set_ylabel(pretty_label(ax.get_ylabel()))
    _prettify_ticklabels(ax)
    ax.tick_params(axis="x", rotation=45)

    ax.set_title(pretty_label(title), pad=10, loc="center")
    fig.tight_layout(pad=1.2)

    out_dir = DIRS.get(folder, DIRS["figures"])
    base = f"fig_{FIG_NO:02d}_{_slugify(filename)}"
    png_path = out_dir / _ensure_suffix(base, ".png")

    fig.savefig(png_path, bbox_inches="tight", dpi=300)

    if show:
        try:
            display(fig)
        except Exception:
            pass

    plt.close(fig)
    return {"fig_no": FIG_NO, "png": png_path}

# =============================
# REGRESSION PNG EXPORT HELPERS (NO individual downloads)
# =============================
def save_model_summary_png(results, filename: str, *, folder: str = "models", fontsize: int = 9, wrap_width: int = 110, show: bool = True):
    """
    Save statsmodels results.summary() text as a PNG in outputs/models (no download).
    """
    global MODEL_NO
    MODEL_NO += 1

    out_dir = DIRS.get(folder, DIRS["models"])
    out_path = out_dir / f"model_{MODEL_NO:02d}_{_slugify(filename)}.png"

    summary_text = results.summary().as_text()
    wrapped = "\n".join(textwrap.fill(line, wrap_width) for line in summary_text.splitlines())

    n_lines = wrapped.count("\n") + 1
    fig_h = max(4, min(40, 0.18 * n_lines))
    fig_w = max(6, min(20, wrap_width / 8.0))

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor("white")
    plt.axis("off")
    plt.text(0, 1, wrapped, va="top", ha="left", family="monospace", fontsize=fontsize)

    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    if show:
        display(fig)
    plt.close(fig)
    return out_path

def save_regression_diagnostics(results, prefix: str, *, folder: str = "models", show: bool = True):
    """
    Save common regression diagnostic plots to PNGs in outputs/models (no download).
    """
    out_dir = DIRS.get(folder, DIRS["models"])
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    fitted = results.fittedvalues
    resid = results.resid

    # 1) Residuals vs Fitted
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(fitted, resid, edgecolor="k", alpha=0.6)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    ax.set_title("Residuals vs Fitted", loc="center")
    fig.tight_layout()
    p = out_dir / f"{_slugify(prefix)}_resid_vs_fitted.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show:
        display(fig)
    plt.close(fig)
    saved["resid_vs_fitted"] = p

    # 2) QQ plot
    fig = sm.qqplot(results.resid, line="45", fit=True)
    fig.set_size_inches(6.5, 6)
    fig.axes[0].set_title("Normal Q-Q", loc="center")
    p = out_dir / f"{_slugify(prefix)}_qq.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show:
        display(fig)
    plt.close(fig)
    saved["qq"] = p

    # 3) Scale-Location
    std_resid = results.get_influence().resid_studentized_internal
    sqrtd = np.sqrt(np.abs(std_resid))
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(fitted, sqrtd, edgecolor="k", alpha=0.6)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Sqrt(|Standardized residuals|)")
    ax.set_title("Scale-Location", loc="center")
    fig.tight_layout()
    p = out_dir / f"{_slugify(prefix)}_scale_location.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show:
        display(fig)
    plt.close(fig)
    saved["scale_location"] = p

    # 4) Influence plot
    fig = plt.figure(figsize=(8, 6))
    sm.graphics.influence_plot(results, ax=fig.add_subplot(111), criterion="cooks")
    fig.axes[0].set_title("Influence Plot (Cook's Distance)", loc="center")
    fig.tight_layout()
    p = out_dir / f"{_slugify(prefix)}_influence.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show:
        display(fig)
    plt.close(fig)
    saved["influence"] = p

    return saved

# -----------------------------
# Docx helper (minimal, NO downloads)
# -----------------------------
def convert_html_to_word(html_path_or_string, out_docx_path):
    out_docx_path = str(out_docx_path)
    if isinstance(html_path_or_string, str) and Path(html_path_or_string).exists():
        with open(html_path_or_string, "r", encoding="utf-8") as f:
            html_text = f.read()
    else:
        html_text = str(html_path_or_string)

    try:
        if Document is not None:
            doc = Document()
            doc.add_paragraph("Converted HTML snapshot (plain text).")
            doc.add_paragraph(html_text[:7000])
            doc.save(out_docx_path)
        else:
            with open(out_docx_path, "w", encoding="utf-8") as f:
                f.write(html_text)
    except Exception:
        with open(out_docx_path, "w", encoding="utf-8") as f:
            f.write(html_text)

    return out_docx_path

# -----------------------------
# Save text output helper (NO downloads)
# -----------------------------
def save_text_output(text, filename, *, folder="models"):
    out_dir = DIRS.get(folder, DIRS["models"])
    out_dir.mkdir(parents=True, exist_ok=True)
    base = _slugify(filename)
    path = out_dir / _ensure_suffix(base, ".txt")
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(text))
    return path

# =============================
# ZIP EXPORT (ONLY downloads the 4 zips)
# Call export_all_zips() at the VERY END of your notebook.
# =============================
def export_all_zips(*, zip_root: Path = BASE_OUT, pause_sec: float = 0.35):
    """
    Creates and downloads 4 zip files:
    - outputs_tables.zip
    - outputs_figures.zip
    - outputs_models.zip
    - outputs_appendix.zip

    This is the ONLY place that triggers downloads.
    """
    if files is None:
        print("Not running in Colab (google.colab.files not available). Zips will be created but not downloaded.")

    zip_root = Path(zip_root)

    to_zip = [
        ("outputs_tables", DIRS["tables"]),
        ("outputs_figures", DIRS["figures"]),
        ("outputs_models", DIRS["models"]),
        ("outputs_appendix", DIRS["appendix"]),
    ]

    made = []
    for base_name, folder_path in to_zip:
        folder_path = Path(folder_path)
        folder_path.mkdir(parents=True, exist_ok=True)

        # make_archive writes base_name + ".zip" in the current working directory unless you give a path
        zip_base = zip_root / base_name
        zip_path = zip_root / f"{base_name}.zip"

        # remove old zip if it exists
        try:
            zip_path.unlink()
        except Exception:
            pass

        shutil.make_archive(str(zip_base), "zip", str(folder_path))
        made.append(zip_path)

    print("Created zip files:")
    for p in made:
        print(f"- {p.as_posix()}")

    # download zips only
    if files is not None:
        for p in made:
            time.sleep(float(pause_sec))
            files.download(str(p))

    return made

print("Output folders ready:")
for k, v in DIRS.items():
    print(f"- {k}: {v.as_posix()}")

print("\nIMPORTANT:")
print("- No individual file downloads will happen.")
print("- At the very end, run: export_all_zips()")

Output folders ready:
- tables: outputs/tables
- figures: outputs/figures
- models: outputs/models
- appendix: outputs/appendix

IMPORTANT:
- No individual file downloads will happen.
- At the very end, run: export_all_zips()


## 2. Load Data (Click choose files, and select the baseball salary.xlsx file)

In [ ]:
from google.colab import files

uploaded = files.upload()
xlsx_name = next(iter(uploaded.keys()))
print("Loaded file:", xlsx_name)

# Read Excel
# If multiple sheets exist, read the first sheet
df = pd.read_excel(xlsx_name)
print("Shape:", df.shape)
display(df.head())

## 3. Quick Data Audit

In [ ]:
# Table A1: First 10 rows (raw input)
tblA1 = display_titled_table(
    df.head(10),
    "First 10 Rows (Raw Input)",
    "raw_first_10",
    folder="appendix",
    index=False,
    # For the raw table, treat SALARY as currency if present
    currency_cols=["SALARY"] if "SALARY" in df.columns else None,
    # counts left unspecified here
    decimals=2
)

# Table A2: Describe summary
desc = df.describe(include="all").transpose()

# Force numeric stats columns to numeric where possible
stat_cols = ["count", "unique", "freq", "mean", "std", "min", "25%", "50%", "75%", "max"]
for c in stat_cols:
    if c in desc.columns:
        desc[c] = pd.to_numeric(desc[c], errors="coerce")

# Blank-out missing values
desc = desc.where(desc.notna(), "")

desc = desc.replace({"nan": "", "NaN": "", "None": ""})

tblA2 = display_titled_table(
    desc,
    "Data Dictionary Summary (describe)",
    "describe_all",
    folder="appendix",
    index=True,
    decimals=2,
    int_cols=[c for c in ["count", "unique", "freq"] if c in desc.columns],
    max_width_in=34.0
)

# Table A3: Missing values
missing = df.isna().sum().sort_values(ascending=False).to_frame("Missing")
tblA3 = display_titled_table(
    missing,
    "Missing Values by Column",
    "missing_by_column",
    folder="appendix",
    index=True,
    decimals=0,  # missing counts are integers
    int_cols=["Missing"]
)

print("Quick audit complete.")

## 4. Data Quality Diagnostics

This section flags potential data entry inconsistencies in LEAGUE, TEAM, and POSITION.
It does not modify the dataset used for statistical analysis.

In [ ]:
from difflib import get_close_matches
import pandas as pd
import numpy as np

# Use the main df from previous cell
df_diag = df.copy()

# Helper for numeric conversion
def _as_numeric(s):
    return pd.to_numeric(s, errors="coerce")

# Ensure columns exist
for c in ["TEAM", "LEAGUE", "DIVISION", "POSITION", "SALARY", "NAME"]:
    if c not in df_diag.columns:
        df_diag[c] = ""

# Normalize
df_diag["TEAM"] = df_diag["TEAM"].astype(str).str.strip().str.replace(" ", "", regex=False)
df_diag["LEAGUE"] = df_diag["LEAGUE"].astype(str).str.strip().str.title()
df_diag["DIVISION"] = df_diag["DIVISION"].astype(str).str.strip().str.title()
df_diag["POSITION"] = df_diag["POSITION"].astype(str).str.strip().str.upper()

# Official valid team list (formerly canonical)
valid_team_league = {
    "Baltimore": "American",
    "Boston": "American",
    "NewYork": None,
    "TampaBay": "American",
    "Toronto": "American",
    "Cleveland": "American",
    "Detroit": "American",
    "Chicago": None,
    "KansasCity": "American",
    "Minnesota": "American",
    "Oakland": "American",
    "Houston": "American",
    "LosAngeles": None,
    "Seattle": "American",
    "Texas": "American",
    "Atlanta": "National",
    "Miami": "National",
    "Philadelphia": "National",
    "Washington": "National",
    "Cincinnati": "National",
    "Pittsburgh": "National",
    "Milwaukee": "National",
    "StLouis": "National",
    "Arizona": "National",
    "Colorado": "National",
    "SanDiego": "National",
    "SanFrancisco": "National",
}

valid_teams = set(valid_team_league.keys())

# Known expected replacements
team_replacements_expected = {
    "California": "LosAngeles",
    "Minneapolis": "Minnesota"
}

legit_positions_set = {"UT", "DH", "LF", "CF", "RF", "SS", "OF", "2B", "1B", "3B", "C"}

def suggest_team_name(team):
    matches = get_close_matches(team, list(valid_teams), n=3, cutoff=0.5)
    return ", ".join(matches)

# Flags
df_diag["TEAM_UNKNOWN"] = ~df_diag["TEAM"].isin(valid_teams)

# ---- TEAM SUGGESTION LOGIC ----
df_diag["TEAM_SUGGESTION"] = ""

# 1️⃣ Known replacements first (ensures California -> LosAngeles always appears)
mask_known = df_diag["TEAM"].isin(team_replacements_expected.keys())
df_diag.loc[mask_known, "TEAM_SUGGESTION"] = df_diag.loc[mask_known, "TEAM"].map(team_replacements_expected)

# 2️⃣ Fuzzy suggestions for remaining unknown teams
mask_unknown_need_suggest = (
    df_diag["TEAM_UNKNOWN"]
    & df_diag["TEAM"].notna()
    & (df_diag["TEAM_SUGGESTION"] == "")
)

df_diag.loc[mask_unknown_need_suggest, "TEAM_SUGGESTION"] = (
    df_diag.loc[mask_unknown_need_suggest, "TEAM"].apply(suggest_team_name)
)

# League mismatch check
def league_is_wrong(row):
    team = row["TEAM"]
    league = row["LEAGUE"]
    expected = valid_team_league.get(team, None)
    if expected is None:
        return False
    return league != expected

df_diag["LEAGUE_MISMATCH"] = df_diag.apply(league_is_wrong, axis=1)
df_diag["POSITION_BAD"] = ~df_diag["POSITION"].isin(legit_positions_set)

# Build issue note
df_diag["ISSUE_NOTE"] = ""

# Known typos
df_diag.loc[mask_known, "ISSUE_NOTE"] += (
    "Team typo (expected: "
    + df_diag["TEAM"].map(team_replacements_expected).astype(str)
    + "). "
)

df_diag.loc[df_diag["TEAM_UNKNOWN"], "ISSUE_NOTE"] += "Team not in valid team list. "
df_diag.loc[df_diag["LEAGUE_MISMATCH"], "ISSUE_NOTE"] += "League mismatch for team. "
df_diag.loc[df_diag["POSITION_BAD"], "ISSUE_NOTE"] += "Position not in valid set. "

# Filter flagged rows
dq_rows = df_diag[
    df_diag["TEAM_UNKNOWN"]
    | df_diag["LEAGUE_MISMATCH"]
    | df_diag["POSITION_BAD"]
].copy()

cols = [c for c in ["NAME", "TEAM", "LEAGUE", "DIVISION", "POSITION", "SALARY"] if c in dq_rows.columns]
dq_rows = dq_rows[cols + ["TEAM_SUGGESTION", "ISSUE_NOTE"]].copy()

if "SALARY" in dq_rows.columns:
    dq_rows["SALARY"] = _as_numeric(dq_rows["SALARY"])

# -----------------------------
# Styling
# -----------------------------
def dq_png_style(df_):
    styles = pd.DataFrame("", index=df_.index, columns=df_.columns)

    if "TEAM" in df_.columns:
        team_unknown = ~df_["TEAM"].astype(str).isin(valid_teams)
        styles.loc[team_unknown, "TEAM"] = "background-color: #fff3cd; font-weight: 700;"

    if "LEAGUE" in df_.columns and "TEAM" in df_.columns:
        expected = df_["TEAM"].map(valid_team_league)
        league_mismatch = expected.notna() & (df_["LEAGUE"] != expected)
        styles.loc[league_mismatch, "LEAGUE"] = "background-color: #f8d7da; font-weight: 700;"

    if "POSITION" in df_.columns:
        pos_bad = ~df_["POSITION"].astype(str).isin(legit_positions_set)
        styles.loc[pos_bad, "POSITION"] = "background-color: #fde2e2; font-weight: 700;"

    # Center TEAM_SUGGESTION
    if "TEAM_SUGGESTION" in df_.columns:
        styles["TEAM_SUGGESTION"] += " text-align: center;"

    # Left align ISSUE_NOTE
    if "ISSUE_NOTE" in df_.columns:
        styles["ISSUE_NOTE"] += " text-align: left;"

    return styles

tblA4 = display_titled_table(
    dq_rows,
    "Data Quality Flags (Reference Only, Not Used in Statistical Analysis)",
    "data_quality_flags",
    folder="appendix",
    currency_cols=["SALARY"] if "SALARY" in dq_rows.columns else None,
    decimals=2,
    index=False,
    extra_style=dq_png_style,
    wrap_cols=["ISSUE_NOTE"],
    wrap_width=72,
    max_width_in=20.0,
    header_height_mult=1.15,
    cell_pad=0.06,
    row_scale=1.05,
)

print("Flagged rows:", dq_rows.shape[0])

## Supplemental Appendix Table: Team Roster Size Summary by League

In [ ]:
league_team_sizes = (
    df.groupby(["LEAGUE", "TEAM"])
    .size()
    .reset_index(name="num_players")
)

league_sizes_summary = league_team_sizes.groupby("LEAGUE")["num_players"].describe()

_ = display_titled_table(
    league_sizes_summary,
    "Team Roster Size Summary by League",
    "roster_size_by_league",
    decimals=0,
    folder="appendix",
    index=True,
)

## Supplemental Appendix Table: Position Counts by Team (with Position Errors)

In [ ]:
import pandas as pd

legit_positions_list = ["UT", "DH", "LF", "CF", "RF", "SS", "OF", "2B", "1B", "3B", "C"]

df_supp = df.copy()
df_supp["TEAM"] = df_supp["TEAM"].astype(str).str.strip().str.replace(" ", "", regex=False)
df_supp["LEAGUE"] = df_supp["LEAGUE"].astype(str).str.strip().str.title()
df_supp["POSITION"] = df_supp["POSITION"].astype(str).str.strip().str.upper()

# Crosstab (League x Team rows, Position columns)
position_by_team = pd.crosstab([df_supp["LEAGUE"], df_supp["TEAM"]], df_supp["POSITION"])
position_by_team.columns = position_by_team.columns.map(str)

# Ensure legit position columns exist
for pos in legit_positions_list:
    if pos not in position_by_team.columns:
        position_by_team[pos] = 0

position_by_team = position_by_team.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)

all_pos_cols = list(position_by_team.columns)
non_legit_cols = sorted([c for c in all_pos_cols if c not in legit_positions_list])

# Add diagnostics columns
position_by_team["Unknown_Positions"] = (
    position_by_team[non_legit_cols]
    .apply(lambda r: ", ".join([c for c in non_legit_cols if r[c] > 0]), axis=1)
    if non_legit_cols else ""
)

position_by_team["Missing_Legit_Count"] = position_by_team[legit_positions_list].eq(0).sum(axis=1)
position_by_team["Unknown_Pos_Count"] = position_by_team[non_legit_cols].sum(axis=1) if non_legit_cols else 0
position_by_team["Total_Players"] = position_by_team[all_pos_cols].sum(axis=1)

# Reorder columns
ordered_cols = non_legit_cols + legit_positions_list + [
    "Total_Players", "Missing_Legit_Count", "Unknown_Pos_Count", "Unknown_Positions"
]
position_by_team = position_by_team[ordered_cols]

# Flatten MultiIndex into columns (League, Team)
position_by_team = position_by_team.reset_index()

# Sort for readability
position_by_team = position_by_team.sort_values(
    ["LEAGUE", "Unknown_Pos_Count", "Missing_Legit_Count", "Total_Players", "TEAM"],
    ascending=[True, False, False, False, True],
)

def style_clean(df_):
    styles = pd.DataFrame("", index=df_.index, columns=df_.columns)

    # Highlight non-legit position columns
    for col in non_legit_cols:
        if col in df_.columns:
            styles[col] = "background-color: #fffbe6;"
            styles.loc[df_[col].gt(0), col] = "background-color: #ffcc80; font-weight: 400;"

    # Highlight missing legit positions (0 counts)
    for col in legit_positions_list:
        if col in df_.columns:
            styles.loc[df_[col].eq(0), col] = "background-color: #ffb3b3;"

    # Totals + summary columns
    if "Total_Players" in df_.columns:
        styles["Total_Players"] = "background-color: #f2f2f2; font-weight: 400;"
    if "Missing_Legit_Count" in df_.columns:
        styles.loc[df_["Missing_Legit_Count"].gt(0), "Missing_Legit_Count"] = "background-color: #f4a3a3; font-weight: 400;"
    if "Unknown_Pos_Count" in df_.columns:
        styles.loc[df_["Unknown_Pos_Count"].gt(0), "Unknown_Pos_Count"] = "background-color: #f6c27a; font-weight: 400;"
    if "Unknown_Positions" in df_.columns:
        styles["Unknown_Positions"] = "background-color: #f2f2f2; font-weight: 400; text-align: center;"

    # Make zeros gray (but do not gray-out key summary columns)
    protected_cols = set(["LEAGUE", "TEAM"] + legit_positions_list + ["Missing_Legit_Count", "Unknown_Pos_Count", "Total_Players"])
    for col in df_.columns:
        if col not in protected_cols and pd.api.types.is_numeric_dtype(df_[col]):
            styles.loc[df_[col].eq(0), col] += "color: #bbbbbb; font-weight: 400;"

    # Left-align League/Team
    if "LEAGUE" in df_.columns:
        styles["LEAGUE"] = "text-align: center;"
    if "TEAM" in df_.columns:
        styles["TEAM"] = "text-align: center;"

    return styles

_ = display_titled_table(
    position_by_team,
    "Position Counts by Team (with Unknown or Missing Position Flags)",
    "position_count_by_team",
    extra_style=style_clean,
    decimals=0,
    index=False,
    folder="appendix",
    wrap_cols=["Unknown_Positions"],
    wrap_width=20,
    max_width_in=16.0,

    # ✅ NEW: wrap the last 4 headers into multiple lines
    header_wrap_map={
        "Total_Players": "Total\nPlayers",
        "Missing_Legit_Count": "Missing\nLegit Count",
        "Unknown_Pos_Count": "Unknown\nPos Count",
        "Unknown_Positions": "Unknown\nPositions",
    },

    # Optional, but usually helps wrapped headers not feel cramped
    header_height_mult=1.45,
)

## 5. Descriptive Analytics Methods

### Method 1: Descriptive Statistics and Distribution Charts

In [ ]:
# Prepare analysis dataset
df_analysis = df.copy()
df_analysis = df_analysis.dropna(subset=["SALARY"]).copy()

df_analysis["LEAGUE"] = df_analysis["LEAGUE"].astype(str).str.strip().str.title()
df_analysis["POSITION"] = df_analysis["POSITION"].astype(str).str.strip().str.upper()
df_analysis["TEAM"] = df_analysis["TEAM"].astype(str).str.strip().str.replace(" ", "", regex=False)

df_analysis["SALARY"] = pd.to_numeric(df_analysis["SALARY"], errors="coerce")
df_analysis = df_analysis.dropna(subset=["SALARY"]).copy()

# Salary is in thousands (per assignment style), log transform
df_analysis["LOG_SALARY"] = np.log(df_analysis["SALARY"])

df_analysis["EXP_BAND"] = pd.cut(
    df_analysis["YR_MAJOR"],
    bins=[-1, 2, 5, 10, 50],
    labels=["0-2", "3-5", "6-10", "11+"],
    include_lowest=True,
)

print("Analysis dataset shape:", df_analysis.shape)

# Method 1: Salary summary table (use currency formatting, salary is in thousands)
summary = df_analysis["SALARY"].describe().to_frame("SALARY").T
summary["mean"] = df_analysis["SALARY"].mean()
summary["median"] = df_analysis["SALARY"].median()
summary["skew"] = df_analysis["SALARY"].skew()

tbl1 = display_titled_table(
    summary,
    "Salary Summary Statistics (Salary in $ thousands)",
    "salary_summary_stats",
    currency_cols=["SALARY", "mean", "median"],
    decimals=2,
    index=False,
    extra_decimal_map={"skew": 3}
)

# Distribution chart: histogram (raw)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
vals = df_analysis["SALARY"].dropna().to_numpy()
ax.hist(vals, bins=30)
ax.set_xlabel("Salary ($ thousands)")
ax.set_ylabel("Number of players")

# Add reference lines
p25, p50, p75 = np.quantile(vals, [0.25, 0.50, 0.75])
ax.axvline(p50, linewidth=1.2, linestyle="--")
ax.text(p50, ax.get_ylim()[1]*0.92, f"Median: {int(round(p50))}", rotation=90, va="top")

save_figure("Salary Distribution (Raw, $ thousands)", "salary_hist_raw", fig=fig, ax=ax)

# Distribution chart: histogram (log)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
vals_log = df_analysis["LOG_SALARY"].dropna().to_numpy()
ax.hist(vals_log, bins=30)
ax.set_xlabel("Log(Salary), Salary in $ thousands")
ax.set_ylabel("Number of players")
save_figure("Salary Distribution (Log Scale)", "salary_hist_log", fig=fig, ax=ax)

# Percentiles table
salary_percentiles = df_analysis["SALARY"].quantile([0.25, 0.5, 0.75, 1.0]).to_frame("Salary")
salary_percentiles.index = ["25th Percentile", "50th Percentile (Median)", "75th Percentile", "Maximum"]

_ = display_titled_table(
    salary_percentiles,
    "Salary Distribution Percentiles (Salary in $ thousands)",
    "salary_percentiles",
    currency_cols=["Salary"],
    decimals=2,
    index=True,
)

### Method 2: Crosstabs and Group Summaries

In [ ]:
# League summary
league_summary = (
    df_analysis.groupby("LEAGUE")["SALARY"]
    .agg(count="count", mean="mean", median="median", std="std")
)

tbl2 = display_titled_table(
    league_summary,
    "Salary Summary by League (Salary in $ thousands)",
    "salary_by_league",
    currency_cols=["mean", "median", "std"],
    int_cols=["count"],
    decimals=2,
    index=True,
)

# Best chart for league comparison: boxplot (handles skew well)
fig, ax = plt.subplots(figsize=(7.0, 4.6))
data = [df_analysis.loc[df_analysis["LEAGUE"] == lg, "SALARY"].dropna() for lg in league_summary.index]
ax.boxplot(data, labels=league_summary.index.astype(str), showfliers=False)
ax.set_xlabel("League")
ax.set_ylabel("Salary ($ thousands)")
save_figure("Salary by League (Boxplot, $ thousands)", "salary_boxplot_by_league", fig=fig, ax=ax)

# Position summary
pos_summary = (
    df_analysis.groupby("POSITION", observed=True)["SALARY"]
    .agg(count="count", mean="mean", median="median", std="std")
    .sort_values("median", ascending=False)
)

tbl4 = display_titled_table(
    pos_summary,
    "Salary Summary by Position (Salary in $ thousands)",
    "salary_by_position",
    currency_cols=["mean", "median", "std"],
    int_cols=["count"],
    decimals=2,
    index=True,
)

# Chart: boxplot for top positions by count (keeps it readable)
pos_counts = df_analysis["POSITION"].value_counts()
top_positions = pos_counts.head(10).index.tolist()

plot_df = df_analysis[df_analysis["POSITION"].isin(top_positions)].copy()
order = (
    plot_df.groupby("POSITION")["SALARY"].median().sort_values(ascending=False).index.tolist()
)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
data = [plot_df.loc[plot_df["POSITION"] == p, "SALARY"].dropna() for p in order]
ax.boxplot(data, labels=order, showfliers=False)
ax.set_xlabel("Position (top 10 by roster count)")
ax.set_ylabel("Salary ($ thousands)")
ax.tick_params(axis="x", rotation=45)
save_figure("Salary by Position (Top 10 Positions, Boxplot, $ thousands)", "salary_boxplot_by_position_top10", fig=fig, ax=ax)

# Experience band summary
exp_summary = (
    df_analysis.groupby("EXP_BAND", observed=True)["SALARY"]
    .agg(count="count", mean="mean", median="median", std="std")
)

_ = display_titled_table(
    exp_summary,
    "Salary Summary by Experience Band (Salary in $ thousands)",
    "salary_by_experience_band",
    currency_cols=["mean", "median", "std"],
    int_cols=["count"],
    decimals=2,
    index=True,
)

# Chart: boxplot by experience band
fig, ax = plt.subplots(figsize=(7.6, 4.8))
band_order = exp_summary.index.astype(str).tolist()
data = [df_analysis.loc[df_analysis["EXP_BAND"].astype(str) == b, "SALARY"].dropna() for b in band_order]
ax.boxplot(data, labels=band_order, showfliers=False)
ax.set_xlabel("Experience band (years in MLB)")
ax.set_ylabel("Salary ($ thousands)")
save_figure("Salary by Experience Band (Boxplot, $ thousands)", "salary_boxplot_by_experience_band", fig=fig, ax=ax)

# Team summary table (keep table, but chart only top 10 for readability)
team_summary = (
    df_analysis.groupby("TEAM", observed=True)["SALARY"]
    .agg(count="count", mean="mean", median="median", std="std")
    .sort_values("median", ascending=False)
)

_ = display_titled_table(
    team_summary,
    "Salary Summary by Team (Salary in $ thousands)",
    "salary_by_team",
    currency_cols=["mean", "median", "std"],
    int_cols=["count"],
    decimals=2,
    index=True,
    wrap_cols=[],  # not needed here
    max_width_in=32.0
)

# Full team median chart (sorted)
sorted_team_medians = team_summary.sort_values("median", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(sorted_team_medians.index.astype(str), sorted_team_medians["median"].values)
ax.set_xlabel("Team (Sorted by Median Salary)")
ax.set_ylabel("Median Salary ($ thousands)")
ax.tick_params(axis="x", rotation=45)

save_figure("Median Salary by Team (All Teams, Sorted)",
            "median_salary_by_team_all",
            fig=fig,
            ax=ax)

fig, ax = plt.subplots(figsize=(12, 6))

data = [df_analysis[df_analysis["TEAM"] == team]["SALARY"]
        for team in sorted_team_medians.index]

ax.boxplot(data, labels=sorted_team_medians.index, showfliers=False)
ax.set_xlabel("Team")
ax.set_ylabel("Salary ($ thousands)")
ax.tick_params(axis="x", rotation=45)

save_figure("Salary Distribution by Team (Boxplot)",
            "salary_boxplot_by_team_all",
            fig=fig,
            ax=ax)

# Roster Size vs Median Salary Scatterplot
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(team_summary["count"], team_summary["median"])
ax.set_xlabel("Roster Size")
ax.set_ylabel("Median Salary ($ thousands)")
save_figure("Roster Size vs Median Salary", "roster_size_vs_median", fig=fig, ax=ax)

# Correlation Heatmap (robust)
corr = df_analysis.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values)

# ticks + labels (attach to THIS ax, not global plt)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels([pretty_label(c) for c in corr.columns], rotation=90)
ax.set_yticklabels([pretty_label(c) for c in corr.columns])

# colorbar attached to the same figure
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

save_figure(
    "Correlation Heatmap",
    "correlation_heatmap",
    fig=fig,
    ax=ax,
    show=True
)

### Method 3: Correlation Analysis

In [ ]:
numeric_cols = [
    "NO_ATBAT", "NO_HITS", "NO_HOME", "NO_RUNS", "NO_RBI", "NO_BB", "YR_MAJOR",
    "CR_ATBAT", "CR_HITS", "CR_HOME", "CR_RUNS", "CR_RBI", "CR_BB"
]

corr_series = (
    df_analysis[numeric_cols + ["SALARY"]]
    .corr(numeric_only=True)["SALARY"]
    .drop("SALARY")
    .sort_values(ascending=False)
)

top_corr = corr_series.head(8).to_frame("Correlation_with_Salary")

_ = display_titled_table(
    top_corr,
    "Top Correlations with Salary",
    "top_correlations",
    corr_cols=["Correlation_with_Salary"],
    decimals=3,
    index=True,
)

fig, ax = plt.subplots(figsize=(9.0, 4.8))
ax.bar(top_corr.index.astype(str), top_corr["Correlation_with_Salary"].values)
ax.set_xlabel("Variable")
ax.set_ylabel("Correlation with Salary")
ax.tick_params(axis="x", rotation=45)
save_figure("Top Correlations with Salary", "top_correlations_bar", fig=fig, ax=ax)

# Best scatter view: log salary vs best correlated variable
best_x = top_corr.index[0]
plot_df = df_analysis[[best_x, "LOG_SALARY"]].dropna().sort_values(best_x)

x = plot_df[best_x].to_numpy()
y = plot_df["LOG_SALARY"].to_numpy()

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.scatter(x, y, alpha=0.65)

m, b = np.polyfit(x, y, 1)
ax.plot(x, m * x + b, linewidth=1.5)

ax.set_xlabel(str(best_x))
ax.set_ylabel("Log(Salary), Salary in $ thousands")
save_figure(f"Log(Salary) vs {best_x}", f"scatter_logsalary_vs_{best_x}", fig=fig, ax=ax)

### Method 4: Two-Sample t-test (League Comparison)

In [ ]:
group_A = df_analysis.loc[df_analysis["LEAGUE"] == "American", "SALARY"].dropna()
group_N = df_analysis.loc[df_analysis["LEAGUE"] == "National", "SALARY"].dropna()

t_stat, p_value = stats.ttest_ind(group_A, group_N, equal_var=False, nan_policy="omit")

ttest_summary = pd.DataFrame({
    "League": ["American", "National"],
    "Count": [group_A.shape[0], group_N.shape[0]],
    "Mean Salary": [group_A.mean(), group_N.mean()],
    "Median Salary": [group_A.median(), group_N.median()],
    "Std Dev": [group_A.std(), group_N.std()],
})

tbl3 = display_titled_table(
    ttest_summary,
    "League Salary Comparison (Welch t-test, Salary in $ thousands)",
    "league_ttest_summary",
    currency_cols=["Mean Salary", "Median Salary", "Std Dev"],
    int_cols=["Count"],
    decimals=2,
    index=False,
    extra_decimal_map={},  # p-value shown separately
)

print("Welch t-test t-statistic:", format_fixed(float(t_stat), 3))
print("Welch t-test p-value:", format_pval(float(p_value)))
print("American mean:", format_fixed(group_A.mean(), 2), "National mean:", format_fixed(group_N.mean(), 2))

# --- Welch t-test (American vs National) ---
group_A = df_analysis.loc[df_analysis["LEAGUE"] == "American", "SALARY"].dropna()
group_N = df_analysis.loc[df_analysis["LEAGUE"] == "National", "SALARY"].dropna()

t_stat, p_value = stats.ttest_ind(group_A, group_N, equal_var=False, nan_policy="omit")

# This table is the league descriptive stats (looks like Salary by League, because it is)
ttest_summary = pd.DataFrame({
    "League": ["American", "National"],
    "Count": [group_A.shape[0], group_N.shape[0]],
    "Mean Salary": [group_A.mean(), group_N.mean()],
    "Median Salary": [group_A.median(), group_N.median()],
    "Std Dev": [group_A.std(), group_N.std()],
})

tbl3 = display_titled_table(
    ttest_summary,
    "League Salary Comparison (Welch t-test, Salary in $ thousands)",
    "league_ttest_summary",
    currency_cols=["Mean Salary", "Median Salary", "Std Dev"],
    int_cols=["Count"],
    decimals=2,
    index=False,
)

# This is the missing piece: a PNG table that actually contains the test outputs
ttest_results = pd.DataFrame({
    "Test": ["Welch Two-Sample t-test"],
    "t-statistic": [t_stat],
    "p-value": [p_value],
})

tbl3b = display_titled_table(
    ttest_results,
    "Welch t-test Results (American vs National)",
    "league_welch_ttest_results",
    decimals=4,
    index=False,
    pval_cols=["p-value"],          # uses your format_pval()
    extra_decimal_map={"t-statistic": 3},
    max_width_in=12.0,
)

### Method 5: ANOVA (Group Differences by Position and Team)

In [ ]:
anova_pos = ols("SALARY ~ C(POSITION)", data=df_analysis).fit()
anova_pos_table = sm.stats.anova_lm(anova_pos, typ=2)
anova_pos_table["eta_sq"] = anova_pos_table["sum_sq"] / anova_pos_table["sum_sq"].sum()

_ = display_titled_table(
    anova_pos_table,
    "ANOVA: Salary by Position (with Eta-Squared)",
    "anova_salary_by_position",
    decimals=3,
    index=True,
    folder="tables",
    extra_decimal_map={"eta_sq": 3},
)

anova_team = ols("SALARY ~ C(TEAM)", data=df_analysis).fit()
anova_team_table = sm.stats.anova_lm(anova_team, typ=2)
anova_team_table["eta_sq"] = anova_team_table["sum_sq"] / anova_team_table["sum_sq"].sum()

_ = display_titled_table(
    anova_team_table,
    "ANOVA: Salary by Team (with Eta-Squared)",
    "anova_salary_by_team",
    decimals=3,
    index=True,
    folder="appendix",
    extra_decimal_map={"eta_sq": 3},
    max_width_in=32.0
)

### Method 6: Multiple Regression (Experience, Position, and Performance)

In [ ]:
model_basic = ols("LOG_SALARY ~ YR_MAJOR + C(POSITION)", data=df_analysis).fit()
model_full = ols("LOG_SALARY ~ YR_MAJOR + C(POSITION) + CR_RBI + CR_HOME + CR_RUNS", data=df_analysis).fit()

print(model_basic.summary())
print(model_full.summary())

save_text_output(model_basic.summary().as_text(), "regression_model_basic", folder="models")
save_text_output(model_full.summary().as_text(), "regression_model_full", folder="models")

reg_compare = pd.DataFrame({
    "Model": ["Model A", "Model B"],
    "R2": [model_basic.rsquared, model_full.rsquared],
    "Adj R2": [model_basic.rsquared_adj, model_full.rsquared_adj],
    "AIC": [model_basic.aic, model_full.aic],
    "BIC": [model_basic.bic, model_full.bic],
})

_ = display_titled_table(
    reg_compare,
    "Regression Model Comparison",
    "regression_model_comparison",
    decimals=3,
    index=False,
)

# Residual diagnostics chart
fitted = model_full.fittedvalues
resid = model_full.resid

fig, ax = plt.subplots(figsize=(7.0, 4.8))
ax.scatter(fitted, resid, alpha=0.65)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Fitted Log(Salary)")
ax.set_ylabel("Residuals")
save_figure("Residuals vs Fitted (Model B)", "residuals_vs_fitted_modelB", fig=fig, ax=ax)

# Export regression statistics into png format

In [ ]:
from pathlib import Path
import shutil
import textwrap
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np

# ---- Helper: save statsmodels summary text as PNG ----
def save_model_summary_png(results, out_path, fontsize=9, wrap_width=110):
    """
    Save statsmodels results.summary() text as a PNG.
    - results: fitted results object returned by sm.OLS(...).fit()
    - out_path: pathlib.Path or string where PNG will be saved
    """
    out_path = Path(out_path)
    summary_text = results.summary().as_text()
    wrapped = "\n".join(textwrap.fill(line, wrap_width) for line in summary_text.splitlines())

    # estimate figure size by number of lines
    n_lines = wrapped.count("\n") + 1
    fig_h = max(4, min(40, 0.18 * n_lines))
    fig_w = max(6, min(20, wrap_width / 8.0))

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.patch.set_facecolor("white")
    plt.axis("off")

    plt.text(0, 1, wrapped, va="top", ha="left", family="monospace", fontsize=fontsize)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, bbox_inches="tight", dpi=300)
    display(fig)
    plt.close(fig)
    return out_path

# ---- Helper: save regression diagnostic plots ----
def save_regression_diagnostics(results, out_dir, prefix="reg", show=False):
    """
    Save common regression diagnostic plots to PNG files.
    - results: fitted statsmodels results object (e.g., ols(...).fit())
    - out_dir: folder path to write PNGs to
    - prefix: filename prefix
    - returns dict of saved file paths
    """
    import statsmodels.api as sm
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    # Residuals vs Fitted
    fig, ax = plt.subplots(figsize=(7,5))
    fitted = results.fittedvalues
    resid = results.resid
    ax.scatter(fitted, resid, edgecolor="k", alpha=0.6)
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Residuals")
    ax.set_title("Residuals vs Fitted")
    fig.tight_layout()
    p = out_dir / f"{prefix}_resid_vs_fitted.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show: display(fig)
    plt.close(fig)
    saved["resid_vs_fitted"] = p

    # QQ plot
    fig = sm.qqplot(results.resid, line="45", fit=True)
    fig.set_size_inches(6.5, 6)
    fig.axes[0].set_title("Normal Q-Q")
    p = out_dir / f"{prefix}_qq.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show: display(fig)
    plt.close(fig)
    saved["qq"] = p

    # Scale-Location
    std_resid = results.get_influence().resid_studentized_internal
    sqrtd = np.sqrt(np.abs(std_resid))
    fig, ax = plt.subplots(figsize=(7,5))
    ax.scatter(fitted, sqrtd, edgecolor="k", alpha=0.6)
    ax.set_xlabel("Fitted values")
    ax.set_ylabel("Sqrt(|Standardized residuals|)")
    ax.set_title("Scale-Location")
    fig.tight_layout()
    p = out_dir / f"{prefix}_scale_location.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show: display(fig)
    plt.close(fig)
    saved["scale_location"] = p

    # Influence plot
    fig = plt.figure(figsize=(8,6))
    sm.graphics.influence_plot(results, ax=fig.add_subplot(111), criterion="cooks")
    fig.tight_layout()
    p = out_dir / f"{prefix}_influence.png"
    fig.savefig(p, bbox_inches="tight", dpi=300)
    if show: display(fig)
    plt.close(fig)
    saved["influence"] = p

    return saved

# ---- 1) Save regression summaries and diagnostics ----
# Replace model_basic/model_full names if your objects are named differently
# Make sure these model objects exist in memory before running this cell.
try:
    # Adjust filenames as you prefer
    mb_path = Path(DIRS["models"]) / "regression_model_basic.png"
    mf_path = Path(DIRS["models"]) / "regression_model_full.png"
    save_model_summary_png(model_basic, mb_path, fontsize=8, wrap_width=120)
    save_model_summary_png(model_full, mf_path, fontsize=8, wrap_width=120)

    # Save diagnostics for the more complete model (or both if you want)
    save_regression_diagnostics(model_full, DIRS["figures"], prefix="model_full", show=False)

    print("Saved regression PNGs to:", DIRS["models"].as_posix())
    print("Saved regression diagnostics to:", DIRS["figures"].as_posix())

except NameError as e:
    print("Regression objects not found (model_basic/model_full). Create or rename accordingly.", e)

# ---- 2) Make zip archives for figures and models ----
# You can zip figures and models (and tables if you want) and download the zips.
zip_figures = Path("outputs") / "figures_archive.zip"
zip_models = Path("outputs") / "models_archive.zip"

# Remove old zips if present
for p in [zip_figures, zip_models]:
    try:
        p.unlink()
    except Exception:
        pass

print("Creating zip archives (figures, models)...")
shutil.make_archive(str(zip_figures.with_suffix("")), "zip", DIRS["figures"])
shutil.make_archive(str(zip_models.with_suffix("")), "zip", DIRS["models"])
print("Zips created:", zip_figures.as_posix(), zip_models.as_posix())


# Run last to export all archive zip folders

In [ ]:
from pathlib import Path
import shutil

# Folders you want zipped
BASE_OUT = Path("outputs")
folders = {
    "tables": BASE_OUT / "tables",
    "figures": BASE_OUT / "figures",
    "models": BASE_OUT / "models",
    "appendix": BASE_OUT / "appendix",
}

# Where zip files will be created
zip_paths = {}

# 1) Create zips (overwrite if they already exist)
for name, folder in folders.items():
    if not folder.exists():
        print(f"Skipping {name}: folder not found -> {folder.as_posix()}")
        continue

    zip_base = BASE_OUT / f"{name}_archive"  # shutil adds .zip
    zip_file = zip_base.with_suffix(".zip")

    # remove old zip if present
    if zip_file.exists():
        zip_file.unlink()

    shutil.make_archive(str(zip_base), "zip", str(folder))
    zip_paths[name] = zip_file
    print(f"Created: {zip_file.as_posix()}")

# 2) Download ONLY the zips (no individual files)
try:
    from google.colab import files

    print("\nStarting zip downloads...")
    for name in ["tables", "figures", "models", "appendix"]:
        if name in zip_paths:
            files.download(str(zip_paths[name]))
except Exception as e:
    print("Auto-download failed, but the zip files were created in outputs/. Error:", e)